# Projekt: Optimaler Standort für eine Elektro-Motorrad Messe in Zürich

Beschreibung Business Case

### Ersteller/in:
<li> Fabian Pfister (T WB-T-CAS-INFE-2-24-17) </li>
<li> Perrine Rotzetter (T WB-T-CAS-INFE-2-24-17) </li>
<li> Stefan Schegg (T WB-T-CAS-INFE-2-24-17) </li>

Anmerkung: Damit das Skript läuft muss ein Ordner "data" im gleichen Ordner wie das Skript existieren. \
Ordner "data": Hier ist die ursprüngliche Datengrundlage Gemeindedaten und Geodaten sowie Resultate vorheriger Skript abgespeichert.

## Importiere Bibliotheken

In [5]:
import requests
import zipfile 
import pandas as pd
from io import BytesIO

## Importiere notwendige Daten
### Energyreporter-Daten

Von Interesse sind hier die Daten rund um die Elektro-Autos und die benötigte Infrastruktur (Ladestationen).

In [6]:
# Definiere Filepath
energyreporter_zip_file_path = "https://opendata.geoimpact.ch/energiereporter/energyreporter_latest.zip"

# HTTP-Anfrage, um die ZIP-Datei herunterzuladen
response = requests.get(energyreporter_zip_file_path)

In [7]:
# ZIP-Datei öffnen
energyreporter_zip = zipfile.ZipFile(BytesIO(response.content))

# Liste, um alle DataFrames zu speichern
dfs_energyreporter = []

# Durch alle Dateien in der ZIP-Datei iterieren
for file_info in energyreporter_zip.infolist():
    # Prüfen, ob die Datei eine .csv Datei ist
    if file_info.filename.endswith('.csv'):
        # CSV-Datei öffnen
        csv_file = energyreporter_zip.open(file_info.filename)
        # CSV-Datei in einen DataFrame einlesen und zur Liste hinzufügen
        df = pd.read_csv(csv_file)
        dfs_energyreporter.append(df)
        # Datei schließen
        csv_file.close()

# ZIP-Datei schließen
energyreporter_zip.close()

In [8]:
# Wähle Datensatz und Spalten welche von Interesse sind
dfs_energyreporter_municipality = dfs_energyreporter[2].loc[:,["bfs_nr", "municipality", "canton", "electric_car_charging_spot_count", "electric_cars_per_charging_spot", "electric_car_charging_spot_last_change"]]

In [9]:
# Rohdaten Zwischenspeichern
dfs_energyreporter_municipality.to_csv("./data/dfs_energyreporter_municipality_raw.csv", index=False)

## Datenbereinigung
### Energyreporter-Daten

In [10]:
# Filterung nach Kanton Zürich
dfs_energyreporter_municipality = dfs_energyreporter_municipality[dfs_energyreporter_municipality["canton"] == "ZH"]

In [11]:
# Hat Zelle ein NaN oder nicht.
disturbed_data = dfs_energyreporter_municipality.notna() #True = kein Nan; False = NaN

# Ersetze NaN mit 999 im Original Datensatz
# Grund: Da der Wert 0 im Datensatz auch so existiert, ist er kein geeigneter Ersatz für NaN.
# Wenn keine Angabe zu electric_cars_per_charging_spot nehmen wir den Wert 999 an. So können diese Zeilen im weiteren ohne Probleme identifiziert und bei Bedarf entfernt werden.
# Zusätzlich ist dies der passende Wert für das spätere Ranking (999 = kleinste Punktzahl).
dfs_energyreporter_municipality["electric_cars_per_charging_spot"].where(disturbed_data["electric_cars_per_charging_spot"], 999, inplace = True)

In [12]:
# Kontrolle das keine weiteren NaN Werte existieren
dfs_energyreporter_municipality[dfs_energyreporter_municipality.isna().any(axis=1)].shape

(0, 6)

## Ausgabe des resultierenden Files
### Energyreporter-Daten

In [13]:
# Speichere einen bereinigte Daten.
dfs_energyreporter_municipality.to_csv("./data/dfs_energyreporter_municipality_cleansed.csv", index=False)

## Importiere notwendige Daten
### Gemeindeporträt Daten

In [35]:
# Definiere Filepath
gmeindedaten_path = "./data/Rohdaten_GemeindeportätZuerich.zip"

In [36]:
# ZIP-Datei öffnen
gemeindedaten_zip = zipfile.ZipFile(gmeindedaten_path)

# Liste, um alle DataFrames zu speichern
dfs_gemeindedaten_gesamt = []

# Durch alle Dateien in der ZIP-Datei iterieren
for file_info in gemeindedaten_zip.infolist():
    # Prüfen, ob die Datei eine .csv Datei ist
    if file_info.filename.endswith('.xlsx'):
        # CSV-Datei öffnen
        excel_file = gemeindedaten_zip.open(file_info.filename)
        # CSV-Datei in einen DataFrame einlesen und zur Liste hinzufügen
        df = pd.read_excel(excel_file)
        dfs_gemeindedaten_gesamt.append(df)
        # Datei schließen
        excel_file.close()

# ZIP-Datei schließen
gemeindedaten_zip.close()

## Auffüllen von Jahren
### Gemeindeporträt Daten
Bestimmte Grössen hatten keine Einträge in 2019-2023. In diesen Fällen wird der letzte Wert für die Jahre 2019 - 2023 übernommen.

Betroffene Files Indizes in df_gemeindedaten_gesamt: 9, 10, 11 und 15

In [37]:
# Erforderliche Jahre
years_necessary = {2019, 2020, 2021, 2022, 2023}

# Gehe alle Files durch
for i in range(1, len(dfs_gemeindedaten_gesamt)): 
    # Einzigartige Werte in der Spalte 'Jahre'. Nutze hierzu ein Set.
    years_unique = set(dfs_gemeindedaten_gesamt[i]['INDIKATOR_JAHR'])
    # Fehlende Jahre ermitteln
    missing_years = years_necessary - years_unique

    # Fehlende Jahre mit den Werten des letzten verfügbaren Jahres füllen
    for year in missing_years:
        # Letzten verfügbaren Eintrag kopieren und Jahr aktualisieren
        last_entry = dfs_gemeindedaten_gesamt[i][dfs_gemeindedaten_gesamt[i]['INDIKATOR_JAHR'] == max(years_unique)].copy()
        last_entry['INDIKATOR_JAHR'] = year
        dfs_gemeindedaten_gesamt[i] = pd.concat([dfs_gemeindedaten_gesamt[i], last_entry], ignore_index=True)

## Füge verschiedene Files Zusammen
### Gemeindeporträt Daten
Hier werden auch die Spaltennamen angepasst mittels einer Funktion

In [38]:
# Funktion, die neue Spaltennamen generiert und die gemergeden Daten sinngemäss umbenannt
def rename_columns(df):
    #Gehe Spalten durch.
    for col in df.columns:
        #Prüfe Anfangsstring des Spaltennamens.
        if col.startswith('INDIKATOR_'):         
            # Diese dre Spaltennamen sollten nicht überschrieben werden.
            if(col.endswith("_y") and col != 'INDIKATOR_NAME_y' and col != "INDIKATOR_JAHR"):
                #setze Variable new_name
                new_name = col.replace('INDIKATOR_', str(df['INDIKATOR_NAME_y'][0]) + " ")
                # Entfernen des Suffixes "_y"
                new_name = new_name.replace('_y', '')
                # Entferne "VALUE"
                new_name = new_name.replace('VALUE', '')
                #Benenne Spalte um.
                df.rename(columns={col: new_name}, inplace=True)
                continue 

        #Prüfe Anfangsstring des Spaltennamens
        if (col.startswith('EINHEIT_KURZ_') and col.endswith("_y")):
            #Lösche Spalte mit Einheit da diese im Namen der entsprechenden Spalte bereits enthalten ist.
            df.drop(col, axis = 1, inplace= True)

    # Lösche unnötige Spalte.
    df.drop("INDIKATOR_NAME_y", axis = 1, inplace= True)
    #Ändere Spaltennamen
    df.rename(columns={"INDIKATOR_NAME_x": "INDIKATOR_NAME", "INDIKATOR_VALUE_x" : "INDIKATOR_VALUE", "EINHEIT_KURZ_x" : "EINHEIT_KURZ" }, inplace=True)
    return df

Zusammensetzen der Files. Hierbei werden die Spaltennamen sinnvoll angepasst mit der zuvor definierten Funktion. Zusätzlich werden Spezialfälle behandelt.

In [39]:
# Initializiere Endfile
dfs_gemeindedaten = rename_columns(pd.merge(dfs_gemeindedaten_gesamt[0], dfs_gemeindedaten_gesamt[1], on=["BFS_NR", "GEBIET_NAME", "INDIKATOR_JAHR"]))

# Merge alle Files und korrigiere Spaltennamen
for i in range(2, len(dfs_gemeindedaten_gesamt)): 
    dfs_gemeindedaten = rename_columns(pd.merge(dfs_gemeindedaten, dfs_gemeindedaten_gesamt[i], on=["BFS_NR", "GEBIET_NAME", "INDIKATOR_JAHR"], how = "left"))

In [40]:
# Korrigiere übriggebliebene Spaltennamen
for col in dfs_gemeindedaten.columns:
            if(col.startswith("INDIKATOR_") and col != 'INDIKATOR_NAME' and col != 'INDIKATOR_JAHR'):
                #setze Variable new_name
                new_name = col.replace('INDIKATOR_', str(dfs_gemeindedaten['INDIKATOR_NAME'][0]) + " ")
                # Entferne "VALUE"
                new_name = new_name.replace('VALUE', '')
                #Benenne Spalte um.
                dfs_gemeindedaten.rename(columns={col: new_name}, inplace=True)
                continue

In [41]:
#Lösche letzten unnötige Spalten
dfs_gemeindedaten.drop("INDIKATOR_NAME", axis = 1, inplace= True)
dfs_gemeindedaten.drop("EINHEIT_KURZ", axis = 1, inplace= True)

#Benenne Spalte INDIKATOR_JAHR um
dfs_gemeindedaten.rename(columns={"INDIKATOR_JAHR": "JAHR"}, inplace=True)

## Datenbereinigung
### Gemeindeporträt Daten

In [42]:
# Hat Zelle ein NaN oder nicht.
disturbed_data = dfs_gemeindedaten.notna() #True = kein Nan; False = NaN

# Ersetze NaN mit '-1' im Original Datensatz
# Grund: Da der Wert 0 im Datensatz auch so existiert, ist er kein geeigneter Ersatz für NaN.
# Wenn keine Angabe nehmen wir den Wert -1 an. So können diese Zeilen im weiteren ohne Probleme identifiziert und bei Bedarf entfernt werden.
dfs_gemeindedaten.where(disturbed_data, -1, inplace = True)

In [43]:
# Kontrolle das keine weiteren NaN Werte existieren
dfs_gemeindedaten[dfs_gemeindedaten.isna().any(axis=1)].shape

(0, 19)

In [44]:
# Läsche identische Spalten
dfs_gemeindedaten = dfs_gemeindedaten.T.drop_duplicates().T

In [45]:
# Einheitliche Zahlenformate inklusive Rename
dfs_gemeindedaten['Ø steuerbares Vermögen natürliche Pers. [1000 Fr.] '] = dfs_gemeindedaten['Ø steuerbares Vermögen natürliche Pers. [1000 Fr.] ']*1000
dfs_gemeindedaten.rename(columns={"Ø steuerbares Vermögen natürliche Pers. [1000 Fr.] ": "Ø steuerbares Vermögen natürliche Pers. [Fr.] "}, inplace=True)

In [47]:
dfs_gemeindedaten

,BFS_NR,GEBIET_NAME,JAHR,Bevölkerung: Anteil 20-39-Jährige [%],Bevölkerung: Anteil 40-64-Jährige [%],Motorräder [Anz.],Motorräder pro 1000 Einwohner [Anz.],Bestand Elektromotor [%],Bevölkerung [Pers.],Ø steuerbares Einkommen natürliche Pers. [Fr.],Ø steuerbares Vermögen natürliche Pers. [Fr.],Erschliessung durch S-Bahn+Bus [% der Einw.],"MIV-Wege Quell-, Ziel- und Binnenverkehr [Anz.]",MIV-Anteil (Modal Split) [%],Motorrad-Neuzulassungen [Anz.],Motorrad-Neuzulasssungen pro 1000 Einw. [Anz.],Neuzulassungen Elektromotor [%],Weg zur nächsten Haltestelle [m]
0,1,Aeugst a.A.,2019,18.5,43.8,237,119.8,1.3,1981,93918,814000000,0.0,5232.0,83.0,12,6.1,9.3,214.0
1,1,Aeugst a.A.,2020,18.7,43.8,263,132.8,2.4,2011,88390,767000000,0.0,5232.0,83.0,16,8.1,20.7,215.0
2,1,Aeugst a.A.,2021,18.6,43.3,258,128.3,3.4,1986,86312,782000000,0.0,5232.0,83.0,14,7.0,12.1,213.0
3,1,Aeugst a.A.,2022,18.4,42.4,265,133.4,5.2,1991,86111,859000000,0.0,5232.0,83.0,12,6.0,36.7,213.0
4,1,Aeugst a.A.,2023,18.2,42.4,252,126.6,7.5,1995,86111,859000000,0.0,5232.0,83.0,4,2.0,37.5,213.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,298,Wiesendangen,2019,21.4,37.0,537,82.5,1.2,6601,74154,431000000,30.1,16844.0,88.0,22,3.4,6.8,289.0
808,298,Wiesendangen,2020,21.2,36.8,545,82.6,1.6,6636,74697,436000000,30.1,16844.0,88.0,40,6.1,9.2,292.0
809,298,Wiesendangen,2021,21.2,36.5,552,83.2,2.5,6659,75281,459000000,30.1,16844.0,88.0,33,5.0,15.2,293.0
810,298,Wiesendangen,2022,21.0,36.6,524,78.7,3.4,6699,73135,453000000,30.1,16844.0,88.0,27,4.1,25.4,291.0


## Ausgabe des resultierenden Files
### Gemeindeporträt Daten

In [48]:
#Zwischenspeicherung
dfs_gemeindedaten.to_csv("./data/dfs_gemeindedaten_komplett.csv", index=False)

In [49]:
dfs_gemeindedaten

,BFS_NR,GEBIET_NAME,JAHR,Bevölkerung: Anteil 20-39-Jährige [%],Bevölkerung: Anteil 40-64-Jährige [%],Motorräder [Anz.],Motorräder pro 1000 Einwohner [Anz.],Bestand Elektromotor [%],Bevölkerung [Pers.],Ø steuerbares Einkommen natürliche Pers. [Fr.],Ø steuerbares Vermögen natürliche Pers. [Fr.],Erschliessung durch S-Bahn+Bus [% der Einw.],"MIV-Wege Quell-, Ziel- und Binnenverkehr [Anz.]",MIV-Anteil (Modal Split) [%],Motorrad-Neuzulassungen [Anz.],Motorrad-Neuzulasssungen pro 1000 Einw. [Anz.],Neuzulassungen Elektromotor [%],Weg zur nächsten Haltestelle [m]
0,1,Aeugst a.A.,2019,18.5,43.8,237,119.8,1.3,1981,93918,814000000,0.0,5232.0,83.0,12,6.1,9.3,214.0
1,1,Aeugst a.A.,2020,18.7,43.8,263,132.8,2.4,2011,88390,767000000,0.0,5232.0,83.0,16,8.1,20.7,215.0
2,1,Aeugst a.A.,2021,18.6,43.3,258,128.3,3.4,1986,86312,782000000,0.0,5232.0,83.0,14,7.0,12.1,213.0
3,1,Aeugst a.A.,2022,18.4,42.4,265,133.4,5.2,1991,86111,859000000,0.0,5232.0,83.0,12,6.0,36.7,213.0
4,1,Aeugst a.A.,2023,18.2,42.4,252,126.6,7.5,1995,86111,859000000,0.0,5232.0,83.0,4,2.0,37.5,213.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807,298,Wiesendangen,2019,21.4,37.0,537,82.5,1.2,6601,74154,431000000,30.1,16844.0,88.0,22,3.4,6.8,289.0
808,298,Wiesendangen,2020,21.2,36.8,545,82.6,1.6,6636,74697,436000000,30.1,16844.0,88.0,40,6.1,9.2,292.0
809,298,Wiesendangen,2021,21.2,36.5,552,83.2,2.5,6659,75281,459000000,30.1,16844.0,88.0,33,5.0,15.2,293.0
810,298,Wiesendangen,2022,21.0,36.6,524,78.7,3.4,6699,73135,453000000,30.1,16844.0,88.0,27,4.1,25.4,291.0
